## **Graph with Pydantic state** 

In [4]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field 

llm = ChatOpenAI(model='gpt-5-mini', temperature=0)

class GraphState(BaseModel):
    topic: str = Field("Topic I want to generate my content for.")
    social_media_platform: str = Field("Social Media where the content is supposed to be posted")
    content: str = Field("Content provided by the LLM for the given topic")
    curated_content: str = Field("Content after being curated for the give social media platform")

def content_gen(state):
    topic  = state.topic
    content = llm.invoke(f"Give me some content about {topic}").content
    state.content = content

    return state

def curator(state):
    content = state.content
    curated_content = llm.invoke(f"Curate the content: {content} for {state.social_media_platform}").content
    state.curated_content = curated_content

    return state  

my_pydantic_graph = StateGraph(GraphState)
my_pydantic_graph.add_node("content_gen", content_gen)
my_pydantic_graph.add_node("curator", curator)

my_pydantic_graph.add_edge(START, "content_gen")
my_pydantic_graph.add_edge("content_gen", "curator")
my_pydantic_graph.add_edge("curator", END)

my_pydantic_graph_compiled = my_pydantic_graph.compile()
my_pydantic_graph_compiled.invoke(
    {
        'topic': "Messi vs Ronaldo",
        'social_media_platform': "Instagram",
        'content': "",
        'curated_content': ""
    }
)

{'topic': 'Messi vs Ronaldo',
 'social_media_platform': 'Instagram',
 'content': "Short overview — why the debate matters\n- Lionel Messi and Cristiano Ronaldo are widely regarded as the two defining footballers of the 21st century. Their differing styles, consistent excellence, trophy hauls and overlapping peak years created a long-running public and statistical rivalry that shaped how fans, pundits and clubs measure greatness.\n\nSnapshot bios\n- Lionel Messi: Argentine forward/creator. Long Barcelona career (came through La Masia), moved to Paris Saint-Germain in 2021, then to Inter Miami in 2023. Renowned for dribbling, vision, playmaking and finishing.\n- Cristiano Ronaldo: Portuguese forward/finisher. Came through Sporting CP, rose to prominence at Manchester United, became a global superstar at Real Madrid, then Juventus, returned to Manchester United, and moved to Al Nassr in 2023. Renowned for athleticism, finishing, aerial power and work-rate.\n\nPlaying styles — how they dif